In [0]:
%sql
SELECT 'patient_bronze' AS table_name, COUNT(*) AS nb FROM chu.bronze.patient
UNION ALL
SELECT 'patient_silver', COUNT(*) FROM chu.silver.patient
UNION ALL
SELECT 'patient_gold', COUNT(*) FROM chu.gold.dim_patient
UNION ALL
SELECT 'consultation_bronze', COUNT(*) FROM chu.bronze.consultation
UNION ALL
SELECT 'consultation_silver', COUNT(*) FROM chu.silver.consultation
UNION ALL
SELECT 'consultation_gold', COUNT(*) FROM chu.gold.fait_consultation;

In [0]:
%sql
SELECT *
FROM chu.gold.dim_patient
ORDER BY patient_sk
LIMIT 5;

In [0]:
%sql
SELECT 'Régions' AS domaine, 'Bronze' AS couche, COUNT(*) AS nb_lignes FROM chu.bronze.cog_region
UNION ALL
SELECT 'Départements', 'Bronze', COUNT(*) FROM chu.bronze.cog_departement
UNION ALL
SELECT 'Régions', 'Silver', COUNT(*) FROM chu.silver.ref_region
UNION ALL
SELECT 'Départements', 'Silver', COUNT(*) FROM chu.silver.ref_departement
UNION ALL
SELECT 'Régions', 'Gold', COUNT(*) FROM chu.gold.dim_region

UNION ALL
SELECT 'Établissements', 'Bronze', COUNT(*) FROM chu.bronze.etablissement_sante
UNION ALL
SELECT 'Établissements', 'Silver', COUNT(*) FROM chu.silver.etablissement
UNION ALL
SELECT 'Établissements', 'Gold', COUNT(*) FROM chu.gold.dim_etablissement

UNION ALL
SELECT 'Activités professionnelles', 'Bronze', COUNT(*) FROM chu.bronze.activite_professionnel_sante
UNION ALL
SELECT 'Activités professionnelles', 'Silver', COUNT(*) FROM chu.silver.pont_professionnel_etablissement
UNION ALL
SELECT 'Activités professionnelles', 'Gold', COUNT(*) FROM chu.gold.pont_professionnel_etablissement

UNION ALL
SELECT 'Décès', 'Bronze', COUNT(*) FROM chu.bronze.deces
UNION ALL
SELECT 'Décès', 'Silver', COUNT(*) FROM chu.silver.deces
UNION ALL
SELECT 'Décès', 'Gold', COUNT(*) FROM chu.gold.fait_deces

UNION ALL
SELECT 'Patients', 'Bronze', COUNT(*) FROM chu.bronze.patient
UNION ALL
SELECT 'Patients', 'Silver', COUNT(*) FROM chu.silver.patient
UNION ALL
SELECT 'Patients', 'Gold', COUNT(*) FROM chu.gold.dim_patient

UNION ALL
SELECT 'Consultations', 'Bronze', COUNT(*) FROM chu.bronze.consultation
UNION ALL
SELECT 'Consultations', 'Silver', COUNT(*) FROM chu.silver.consultation
UNION ALL
SELECT 'Consultations', 'Gold', COUNT(*) FROM chu.gold.fait_consultation

UNION ALL
SELECT 'Hospitalisations', 'Bronze', COUNT(*) FROM chu.bronze.hospitalisation
UNION ALL
SELECT 'Hospitalisations', 'Silver', COUNT(*) FROM chu.silver.hospitalisation
UNION ALL
SELECT 'Hospitalisations', 'Gold', COUNT(*) FROM chu.gold.fait_hospitalisation

UNION ALL
SELECT 'Diagnostics', 'Bronze', COUNT(*) FROM chu.bronze.diagnostic
UNION ALL
SELECT 'Diagnostics', 'Silver', COUNT(*) FROM chu.silver.diagnostic
UNION ALL
SELECT 'Diagnostics', 'Gold', COUNT(*) FROM chu.gold.dim_diagnostic

UNION ALL
SELECT 'Professionnels de santé', 'Bronze', COUNT(*) FROM chu.bronze.professionnel_de_sante
UNION ALL
SELECT 'Professionnels de santé', 'Silver', COUNT(*) FROM chu.silver.professionnel
UNION ALL
SELECT 'Professionnels de santé', 'Gold', COUNT(*) FROM chu.gold.dim_professionnel

UNION ALL
SELECT 'Spécialités', 'Bronze', COUNT(*) FROM chu.bronze.specialites
UNION ALL
SELECT 'Spécialités', 'Silver', COUNT(*) FROM chu.silver.specialite;

In [0]:
%sql
SELECT 'Satisfaction Silver' AS table_name, COUNT(*) AS nb_lignes
FROM chu.silver.satisfaction_esatis
UNION ALL
SELECT 'Satisfaction Gold', COUNT(*)
FROM chu.gold.fait_satisfaction;

In [0]:
%sql
SELECT *
FROM chu.gold.dim_region
ORDER BY code_region
LIMIT 10;

In [0]:
%sql
SELECT
    r.code_region,
    r.libelle_region AS libelle_region_silver,
    r.type_zone,
    r.source,
    g.region_sk,
    g.libelle_region AS libelle_region_gold
FROM chu.silver.ref_region r
LEFT JOIN chu.gold.dim_region g
    ON r.code_region = g.code_region
ORDER BY r.code_region;

In [0]:
%sql
SHOW TABLES IN chu.silver;

In [0]:
%sql
DESCRIBE TABLE chu.silver.ref_departement;

In [0]:
%sql
SELECT
    d.code_departement,
    d.libelle_departement,
    d.code_region,
    r.libelle_region,
    d.sourceb
FROM chu.silver.ref_departement d
LEFT JOIN chu.silver.ref_region r
    ON d.code_region = r.code_region
ORDER BY d.code_region, d.code_departement;

In [0]:
%sql
DESCRIBE TABLE chu.silver.etablissement;

In [0]:
%sql
DESCRIBE TABLE chu.gold.dim_etablissement;

In [0]:
%sql
SELECT
    s.identifiant_organisation,
    s.finess_geo,
    s.finess_juridique,
    s.code_commune,
    s.code_departement,
    s.code_region,
    g.libelle_region,
    CASE
        WHEN s.code_region IS NULL THEN 'Région non renseignée'
        WHEN g.code_region IS NULL THEN 'Région absente de Gold'
        ELSE 'Rattachement effectué'
    END AS controle_region,
    CASE
        WHEN s.finess_geo IS NULL
             AND s.type_structure = 'F' THEN 'FINESS manquant'
        WHEN s.type_structure = 'F'
             AND LENGTH(s.finess_geo) <> 9 THEN 'FINESS à vérifier'
        ELSE 'Identifiant non signalé'
    END AS controle_finess
FROM chu.silver.etablissement s
LEFT JOIN chu.gold.dim_etablissement g
    ON s.identifiant_organisation = g.identifiant_organisation
ORDER BY s.code_region, s.code_departement
LIMIT 50;

In [0]:
%sql
SELECT
    COUNT(*) AS total_etablissements,
    SUM(CASE WHEN code_region IS NOT NULL THEN 1 ELSE 0 END)
        AS avec_region,
    SUM(CASE WHEN code_region IS NULL THEN 1 ELSE 0 END)
        AS sans_region,
    SUM(CASE WHEN type_structure = 'F'
              AND finess_geo IS NOT NULL THEN 1 ELSE 0 END)
        AS avec_finess_geo,
    SUM(CASE WHEN type_structure = 'F'
              AND finess_geo IS NULL THEN 1 ELSE 0 END)
        AS sans_finess_geo
FROM chu.silver.etablissement;

In [0]:
%sql
SELECT
    COUNT(*) AS total_etablissements,
    SUM(CASE WHEN code_region IS NOT NULL THEN 1 ELSE 0 END)
        AS avec_region,
    SUM(CASE WHEN code_region IS NULL THEN 1 ELSE 0 END)
        AS sans_region,
    SUM(CASE WHEN type_structure = 'F'
              AND finess_geo IS NOT NULL THEN 1 ELSE 0 END)
        AS avec_finess_geo,
    SUM(CASE WHEN type_structure = 'F'
              AND finess_geo IS NULL THEN 1 ELSE 0 END)
        AS sans_finess_geo
FROM chu.silver.etablissement;

In [0]:
%sql
SELECT
    COUNT(*) AS nombre_lignes,
    COUNT(DISTINCT professionnel_sk) AS professionnels_distincts,
    COUNT(DISTINCT etablissement_sk) AS etablissements_distincts
FROM chu.gold.pont_professionnel_etablissement;

In [0]:
%sql
SELECT
    COUNT(*) AS total_activites,
    SUM(CASE WHEN identifiant_organisation IS NULL THEN 1 ELSE 0 END)
        AS sans_identifiant_organisation,
    SUM(CASE WHEN identifiant_organisation IS NOT NULL THEN 1 ELSE 0 END)
        AS avec_identifiant_organisation
FROM chu.silver.activite_professionnel_sante;

In [0]:
%sql
DESCRIBE TABLE chu.silver.deces;

In [0]:
%sql
SELECT
    COUNT(*) AS total_deces,
    MIN(date_deces) AS premiere_date,
    MAX(date_deces) AS derniere_date,
    SUM(CASE WHEN code_region IS NOT NULL THEN 1 ELSE 0 END)
        AS deces_avec_region,
    SUM(CASE WHEN code_region IS NULL THEN 1 ELSE 0 END)
        AS deces_sans_region,
    ROUND(
        100.0 * SUM(CASE WHEN code_region IS NOT NULL THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS taux_rattachement_region
FROM chu.silver.deces;

In [0]:
%sql
SELECT
    COUNT(*) AS total_deces,
    MIN(date_deces) AS premiere_date,
    MAX(date_deces) AS derniere_date,
    SUM(CASE WHEN code_region IS NOT NULL THEN 1 ELSE 0 END)
        AS deces_avec_region,
    SUM(CASE WHEN code_region IS NULL THEN 1 ELSE 0 END)
        AS deces_sans_region,
    ROUND(
        100.0 * SUM(CASE WHEN code_region IS NOT NULL THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS taux_rattachement_region
FROM chu.silver.deces;

In [0]:
%sql
DESCRIBE TABLE chu.gold.fait_deces;

In [0]:
%sql
DESCRIBE TABLE chu.gold.fait_deces;

In [0]:
%sql
SELECT
    COUNT(*) AS nombre_deces,
    COUNT(DISTINCT date_sk) AS nombre_dates_distinctes,
    MIN(date_sk) AS premiere_date_sk,
    MAX(date_sk) AS derniere_date_sk,
    COUNT(DISTINCT region_sk) AS nombre_regions,
    COUNT(DISTINCT sexe) AS nombre_categories_sexe,
    SUM(nb_deces) AS total_deces_comptabilises
FROM chu.gold.fait_deces;

In [0]:
%sql
SELECT
    current_database() AS base_actuelle,
    current_schema() AS schema_actuel,
    current_user AS utilisateur;


In [0]:
%sql
SHOW SCHEMAS IN chu;

In [0]:
%sql
SHOW TABLES IN chu.bronze;

In [0]:
%sql
DESCRIBE TABLE chu.bronze.patient;

In [0]:
%sql
DESCRIBE TABLE chu.bronze.consultation;

In [0]:
%sql
DESCRIBE TABLE chu.bronze.professionnel_de_sante;